# Task 3: Build a Chatbot using Hugging Face Transformers
**Data Science Internship – February 2026 | Innomatics Research Labs**

---

## Objective
Build a simple conversational chatbot using a pre-trained transformer model from Hugging Face that can interact with users and generate meaningful responses.

## Tools & Technologies
- Python
- Hugging Face Transformers
- PyTorch
- Jupyter Notebook

---

## Step 1: Install Required Libraries

Install the Hugging Face `transformers` library and `torch` if not already installed.

In [1]:
# Install required libraries
# Run this cell only once (you can skip if already installed)
!pip install transformers torch --quiet


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Import Libraries

We import the necessary libraries from Hugging Face Transformers and PyTorch.

In [2]:
# Import core libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings

# Suppress unnecessary warnings for cleaner output
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")

C:\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully!
PyTorch version: 2.11.0+cpu


## Step 3: Load Pre-trained Model

We use **Microsoft's DialoGPT-medium** — a transformer model specifically fine-tuned for **multi-turn conversational dialogue**.

- **Model**: `microsoft/DialoGPT-medium`
- **Architecture**: GPT-2 based
- **Specialty**: Trained on 147M conversation-like exchanges from Reddit

> First run will download the model (~1.5 GB). Subsequent runs use the cached version.

In [3]:
# -----------------------------------------------
# MODEL LOADING
# Using DialoGPT-medium for conversational tasks
# -----------------------------------------------

MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"Loading tokenizer from: {MODEL_NAME}")
# Tokenizer converts text to token IDs the model understands
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loading model from: {MODEL_NAME}")
# AutoModelForCausalLM loads a causal language model for text generation
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Set model to evaluation mode (disables dropout layers)
model.eval()

print("\nModel and Tokenizer loaded successfully!")
print(f"Total Model Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading tokenizer from: microsoft/DialoGPT-medium


Loading model from: microsoft/DialoGPT-medium


Loading weights: 100%|█████████████████████████████████████████████████████████████| 293/293 [00:00<00:00, 6571.44it/s]



Model and Tokenizer loaded successfully!
Total Model Parameters: 354,823,168


## Step 4: Define the Response Generation Function

This function handles:
- Encoding user input into tokens
- Appending conversation history for context
- Generating a model response
- Decoding tokens back to readable text

In [4]:
def generate_response(user_input, chat_history_ids=None, max_length=1000):
    """
    Generate a chatbot response for the given user input.

    Parameters:
    -----------
    user_input       : str    – The message typed by the user
    chat_history_ids : tensor – Token IDs of previous conversation turns
    max_length       : int    – Maximum total token length for context window

    Returns:
    --------
    response         : str    – The chatbot's generated reply
    chat_history_ids : tensor – Updated conversation history including this turn
    """

    # Step 1: Encode user input and append EOS token (end-of-sequence marker)
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors='pt'   # Return as a PyTorch tensor
    )

    # Step 2: Append new input to conversation history for multi-turn context
    if chat_history_ids is not None:
        # Concatenate previous history with new user input
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        # First turn: no prior history
        bot_input_ids = new_input_ids

    # Step 3: Generate response from the model
    with torch.no_grad():  # Disable gradient computation (not needed for inference)
        chat_history_ids = model.generate(
            bot_input_ids,
            max_length=max_length,                # Max total tokens in context
            pad_token_id=tokenizer.eos_token_id,  # Use EOS as padding token
            no_repeat_ngram_size=3,               # Avoid repeating 3-word sequences
            do_sample=True,                       # Enable probabilistic sampling
            top_k=50,                             # Keep only top 50 probable next tokens
            top_p=0.95,                           # Nucleus sampling: cumulative prob 95%
            temperature=0.75                      # Lower = more focused, higher = more creative
        )

    # Step 4: Decode only the newly generated tokens (skip the input portion)
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True  # Remove EOS and padding tokens from output
    )

    return response, chat_history_ids


print("✅ generate_response() function defined successfully!")

✅ generate_response() function defined successfully!


## Step 5: Run the Interactive Chatbot

The chatbot runs in a loop:
- Accepts user input from console
- Generates and displays a response
- Maintains conversation context across turns
- Exits cleanly when user types `exit` or `quit`

> **Note**: This cell uses `input()` — an input box will appear below the cell in Jupyter. Type your message there and press Enter.

In [5]:
def run_chatbot():
    """
    Main chatbot loop.
    Accepts user input and generates responses continuously
    until the user types 'exit' or 'quit'.
    """

    # Initialize conversation history as None (empty at start)
    chat_history_ids = None
    turn_count = 0  # Count conversation turns

    # ── Welcome Banner ──────────────────────────────────────────────────────
    print("=" * 60)
    print("        🤖 AI Chatbot — Powered by DialoGPT")
    print("=" * 60)
    print("  Type your message and press Enter to chat.")
    print("  Type  'exit'  or  'quit'  to end the session.")
    print("-" * 60)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("-" * 60)

    # ── Main Conversation Loop ───────────────────────────────────────────────
    while True:

        # Read user input from console
        user_input = input("You: ").strip()

        # Gracefully handle empty input
        if not user_input:
            print("Chatbot: Please type something so I can help you! 😊")
            continue

        # Exit condition: check for 'exit' or 'quit' (case-insensitive)
        if user_input.lower() in ["exit", "quit"]:
            print("-" * 60)
            print("Chatbot: Thank you for chatting with me! Have a great day! 👋")
            print(f"📊 Total conversation turns: {turn_count}")
            print("=" * 60)
            break  # Exit the loop and end the session

        # Generate response using the transformer model
        response, chat_history_ids = generate_response(user_input, chat_history_ids)
        turn_count += 1

        # Display chatbot's response
        print(f"Chatbot: {response}")
        print("-" * 60)

        # Reset history every 5 turns to avoid token length overflow
        # DialoGPT has a 1024 token limit — resetting prevents truncation errors
        if turn_count % 5 == 0:
            chat_history_ids = None
            print("[ℹ️  Context reset after 5 turns to maintain response quality]")
            print("-" * 60)


# ── Launch the Chatbot ───────────────────────────────────────────────────────
run_chatbot()

        🤖 AI Chatbot — Powered by DialoGPT
  Type your message and press Enter to chat.
  Type  'exit'  or  'quit'  to end the session.
------------------------------------------------------------
Chatbot: Hello! I am your AI assistant. How can I help you today?
------------------------------------------------------------


You:  what is capital of USA


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Chatbot: USA is the capital of America.
------------------------------------------------------------


You:  what is full form of AI


Chatbot: I think this is a good one
------------------------------------------------------------


You:  exit


------------------------------------------------------------
Chatbot: Thank you for chatting with me! Have a great day! 👋
📊 Total conversation turns: 2


## Step 6: Demo — Automated Conversation (No Manual Input Required)

This cell runs a **preset list of questions automatically** to demonstrate chatbot behavior. Useful for showing output in GitHub without needing interactive input.

In [ ]:
# -------------------------------------------------------
# AUTOMATED DEMO
# Runs preset questions without requiring manual input.
# Ideal for showing output in GitHub/Colab.
# -------------------------------------------------------

demo_questions = [
    "Hello",
    "What is Artificial Intelligence?",
    "Tell me about machine learning.",
    "Who created Python?",
    "What are transformers in NLP?",
    "Thank you"
]

print("=" * 60)
print("   🤖 DEMO — Automated Chatbot Conversation")
print("=" * 60)
print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
print("-" * 60)

# Reset chat history for clean demo
chat_history_ids = None

# Loop through preset demo questions
for question in demo_questions:
    print(f"You: {question}")

    # Generate response for each question
    response, chat_history_ids = generate_response(question, chat_history_ids)

    print(f"Chatbot: {response}")
    print("-" * 60)

# Simulate exit
print("You: exit")
print("Chatbot: Thank you for chatting with me! Have a great day! 👋")
print("=" * 60)

   🤖 DEMO — Automated Chatbot Conversation
Chatbot: Hello! I am your AI assistant. How can I help you today?
------------------------------------------------------------
You: Hello
Chatbot: Hello! :D
------------------------------------------------------------
You: What is Artificial Intelligence?
Chatbot: A computer program.
------------------------------------------------------------
You: Tell me about machine learning.
Chatbot: What do you mean?
------------------------------------------------------------
You: Who created Python?
Chatbot: Who made the Matrix?
------------------------------------------------------------
You: What are transformers in NLP?
Chatbot: They were made to be.
------------------------------------------------------------
You: Thank you


## Step 7: Pipeline Summary & Key Concepts

### Expected Pipeline Flow

```
┌──────────────────────────────────────────────────────┐
│               CHATBOT PIPELINE FLOW                  │
│                                                      │
│  User Input                                          │
│      │                                               │
│      ▼                                               │
│  Tokenizer  (text → token IDs + EOS token)           │
│      │                                               │
│      ▼                                               │
│  Append Chat History (multi-turn context)            │
│      │                                               │
│      ▼                                               │
│  DialoGPT Model (input IDs → response token IDs)     │
│      │                                               │
│      ▼                                               │
│  Decode Response (token IDs → readable text)         │
│      │                                               │
│      ▼                                               │
│  Display Output → Loop Until 'exit' / 'quit'         │
└──────────────────────────────────────────────────────┘
```

---

### Key Concepts Covered

| Concept | Description |
|---|---|
| **Transformer Architecture** | Self-attention based model for sequence tasks |
| **DialoGPT** | GPT-2 fine-tuned on 147M Reddit conversations |
| **Tokenization** | Converts raw text to numerical token IDs |
| **Causal Language Model** | Predicts the next token given all previous tokens |
| **Top-k Sampling** | Sample from top k most probable next tokens |
| **Top-p (Nucleus) Sampling** | Sample from smallest token set covering p probability |
| **Temperature** | Scales logits — lower = focused, higher = creative |
| **Chat History** | Concatenated token IDs passed for multi-turn context |

---

## Summary

This notebook successfully demonstrates:
1. **Loading** a pre-trained DialoGPT transformer model from Hugging Face Hub
2. **Tokenizing** user input and maintaining multi-turn conversation history
3. **Generating** contextual responses using Top-k / Top-p / Temperature sampling
4. **Building** a continuous conversational loop with proper exit handling
5. **Applying** best practices for token length management (context reset)